<h1>github link : </h1>
https://github.com/fizzahussain/UNO-3Player-AIvsHuman

- its a private repository because there were no instructions and in PF project we were old if its public there are chances of submission being flagged as plagarism

<h1>CODE:</h1>

In [158]:
import random
import copy

In [159]:
COLOURS = ['Red', 'Blue', 'Green', 'Yellow']
NUMBERS = list(range(10))         
PLAYERS = ['p1', 'p2', 'p3']  
PLAYER_NAMES = {'p1': 'P1 (Minimax Defensive)',
    'p2': 'P2 (Expectimax Offensive)',
    'p3': 'P3',
}

DEFAULT_SEED = 42

In [160]:
class Card:# single uno card

    def __init__(self, colour, value):
        self.colour = colour
        self.value = value           

        
    def __repr__(self):
        return f"{self.colour} {self.value}"

    def __eq__(self, other):
        return (isinstance(other, Card)and self.colour == other.colour and self.value == other.value)

    def __hash__(self):
        return hash((self.colour, self.value))

    def __deepcopy__(self, memo):
        #colour value dono fixed tu no need to recurse
        return Card(self.colour, self.value)



    def matches(self, top: 'Card'): #same colour or number
        return self.colour == top.colour or self.value == top.value
        
    def is_skip(self):
        return self.value == 'Skip'

#DEBUG
c1 = Card('Red' , 8)
print(c1.is_skip())

c2 = Card('Yellow ', 'Skip')
print(c2.is_skip())

c1.__repr__()

False
True


'Red 8'

In [161]:
def deck_generator(): #total 4×11 so 44cards 0 say 9 sab mai and 1 skip bhi so 11 

    deck = []
    for colour in COLOURS:
        for num in NUMBERS:                  
            deck.append(Card(colour, num))
        deck.append(Card(colour, 'Skip'))   

        
    random.shuffle(deck)
    return deck



#DEBUG
#deck= deck_generator()
#print(deck)
#len(deck)

In [162]:
def get_valid_moves(hand, top_card):
    return [card for card in hand if card.matches(top_card)]

#DEBUG
hand = [Card('Red', 6) , Card('Blue', 3), Card('Yellow', 4) , Card('Yellow', 'Skip') , Card('Green', 'Skip') ]
top = Card('Yellow', 6)
get_valid_moves(hand , top)

[Red 6, Yellow 4, Yellow Skip]

In [163]:
def apply_move(state, move):#player key is Player Names ki keys p1 ,p2 , p3

    player_id, card = move
    
    state_updated = copy.deepcopy(state)#deep copy takay original wala is not mutated
    skip_player= None

    if card is None:#card nai hai valid also reshuffle already played ya discards into deck agar deck hi khali hogaya
        if not state_updated['deck'] and state_updated.get('discards'):
            state_updated['deck'] = state_updated['discards']
            random.shuffle(state_updated['deck'])
            state_updated['discards'] = []

        
        if state_updated['deck']:
            drawn = state_updated['deck'].pop()
            state_updated[player_id].append(drawn)
       

    else:
        hand = state_updated[player_id]
        for i, c in enumerate(hand):
            if c == card:
                hand.pop(i)
                break

        if 'discards' not in state_updated:#purana top ko discard mai 
            state_updated['discards'] = []
            
        state_updated['discards'].append(state_updated['top_card'])

        state_updated['top_card'] = card


        
        if card.is_skip():
            index = PLAYERS.index(player_id)
            skip_player = PLAYERS[(index + 1) % len(PLAYERS)]


    
    return state_updated, skip_player

In [164]:
def evaluate(state, player_id, strategy= 'defensive'):
    opp_keys = [k for k in PLAYERS if k != player_id]
    
    cai = len(state[player_id])
    copp = sum(len(state[k]) for k in opp_keys) / 2.0
    s = sum(1 for c in state[player_id] if c.is_skip())

    if strategy == 'defensive':#penalise own hand harder and reward skips highly
        return 50.0 - 6.0 * cai + 2.0 * copp + 4.0 * s
        
    else:#moderate self-penalty adn strongly reward opponent burden
        return 50.0 - 5.0 * cai + 3.0 * copp + 2.0 * s

In [165]:
class TreeNode:
    
    def __init__(self, label, node_type, score = None):
        self.label = label
        self.node_type = node_type   # max , min ,chance,opp,leaf, terminal pruned
        self.children = []
        self.score = score
        


    
    def add_child(self, child: 'TreeNode'):
        self.children.append(child)


def _node_display(node):
    t  = node.node_type
    if node.score is not None:
        sc = f"({node.score:+.1f})"
    else:
        sc = ""
    
    if t == "MAX":
        tag = node.label.split(" |")[0].strip()
        return f"{tag}-AI {sc}".strip()
    if t == "MIN":
        tag = node.label.split(" |")[0].strip()
        return f"{tag}-MIN {sc}".strip()
    if t == "OPP":
        tag = node.label.split(" |")[0].strip()
        return f"{tag}-OPP {sc}".strip()
        
    if t == "ACTION":
        return f"{node.label} {sc}".strip()
    if t == "CHANCE":
        if len(node.label) <= 14:
            label = node.label
        else:
            label = "CHANCE"
        return f"{label} {sc}".strip()
    if t == "LEAF":
        return f"LEAF {sc}".strip()
    if t == "TERMINAL":
        return node.label
    if t == "PRUNED":
        return "[pruned]"
        
    return f"{node.label} {sc}".strip()


def _subtree_min_width(node, max_depth, cur_depth=0, gap=3):
    label_w = len(_node_display(node))

    if cur_depth < max_depth:
        visible = node.children
    else:
        visible = []
    if not visible:
        return label_w
        
    child_widths = [_subtree_min_width(c, max_depth, cur_depth+1, gap)for c in visible]
    children_total = sum(child_widths) + gap * (len(child_widths) - 1)
    
    return max(label_w, children_total)


def _assign_positions(node, x_left, width, depth, max_depth, gap, out):
    center = x_left + width // 2
    out[id(node)] = (center, depth)

    if depth < max_depth:
        visible = node.children
    else:
        visible = []
        
    if not visible:
        return
    child_widths = [_subtree_min_width(c, max_depth, depth+1, gap) for c in visible]
    
    total = sum(child_widths) + gap * (len(child_widths) - 1)
    start = x_left + max(0, (width - total) // 2)
    cx = start
    
    for child, cw in zip(visible, child_widths):
        _assign_positions(child, cx, cw, depth+1, max_depth, gap, out)
        cx += cw + gap


def _collect_by_depth(node, depth, max_depth, result):
    result.setdefault(depth, []).append(node)
    
    if depth < max_depth:
        for child in node.children:
            _collect_by_depth(child, depth+1, max_depth, result)


def print_tree(root, max_depth=3, gap=3):
    total_w  = _subtree_min_width(root, max_depth, 0, gap) + 6
    positions = {}
    _assign_positions(root, 0, total_w, 0, max_depth, gap, positions)
    
    by_depth = {}
    _collect_by_depth(root, 0, max_depth, by_depth)
    
    canvas_w = total_w + 10

    for depth in sorted(by_depth.keys()):
        nodes_at_level = by_depth[depth]

        
        row = [' '] * canvas_w
        for node in nodes_at_level:
            if id(node) not in positions:
                continue
            cx, _ = positions[id(node)]
            label   = _node_display(node)
            start = cx - len(label) // 2
            for i, ch in enumerate(label):
                pos = start + i
                if 0 <= pos < canvas_w:
                    row[pos] = ch
        print(''.join(row).rstrip())

        
        if depth == max_depth:# truncation hints
            hint_row = [' '] * canvas_w
            needed   = False
            for node in nodes_at_level:
                if node.children and id(node) in positions:
                    cx, _  = positions[id(node)]
                    hint   = f"({len(node.children)} more)"
                    start  = cx - len(hint) // 2
                    for i, ch in enumerate(hint):
                        pos = start + i
                        if 0 <= pos < canvas_w and hint_row[pos] == ' ':
                            hint_row[pos] = ch
                    needed = True
            if needed:
                print(''.join(hint_row).rstrip())
            break

       
        conn = [' '] * canvas_w #row  /  |  \
        for node in nodes_at_level:
            if id(node) not in positions:
                continue
            px, _ = positions[id(node)]
            visible = node.children if depth < max_depth else []
            if not visible:
                continue
            child_centers = [positions[id(ch)][0] for ch in visible if id(ch) in positions]
            if not child_centers:
                continue
                
            left, right = min(child_centers), max(child_centers)
            
            if left < right:
                for x in range(left+1, right):
                    if conn[x] == ' ':
                        conn[x] = '_'
            for cc in child_centers:
                if cc == px:
                    conn[cc] = '|'
                elif cc < px:
                    conn[cc] = '/'
                else:
                    conn[cc] = '\\'

                    
        print(''.join(conn).rstrip())


def print_tree_vertical(node, prefix="", is_root=True, is_last=True, max_depth=3, cur_depth=0):
    if node.score is not None:
        scorestr = f" [{node.score:+.1f}]"
    else:
        scorestr = " "
    if is_root:
        print(f"[{node.node_type:8s}] {node.label}{scorestr}")
        childpre = ""
        
    else:
        if is_last:
            conn = "|__ "
        else:
            conn = "|-- "
        print(f"{prefix}{conn}[{node.node_type:8s}] {node.label}{scorestr}")
        
        if is_last:
            childpre = prefix + "    "
        else:
            childpre = prefix + "|   "

    print(f"Depth: {cur_depth} -> {node.label}")
        
    if cur_depth >= max_depth:
        if node.children:
            print(f"{childpre}... ({len(node.children)} children not shown)")
        return


    
    
    for i, child in enumerate(node.children):
        print_tree_vertical(child, childpre, is_root=False,is_last=(i == len(node.children) - 1), max_depth=max_depth, cur_depth=cur_depth + 1)
        


def print_tree_smart(root, max_depth=3, gap=3):
    def has_big_branch(node):
        if len(node.children) > 6:
            return True
        return any(has_big_branch(child) for child in node.children)

    if has_big_branch(root):
        print("\n......Large tree detected using vertical view.......\n")
        print_tree_vertical(root, max_depth=max_depth)
    else:
        print_tree(root, max_depth=max_depth, gap=gap)
        
        
print("complete process of ai decisions above")



complete process of ai decisions above


In [166]:
def minimax(state,depth, player_id, current_player,skipped, alpha = float('-inf'), beta = float('inf'), tree_build = True):
 
    for p in PLAYERS:
        if len(state[p]) == 0:
            s = (1000.0 + depth) if p == player_id else (-1000.0 - depth)  
            node = TreeNode(f"WIN({p.upper()})", "TERMINAL", s) if tree_build else None
            
            return s, None, node

    
    if depth == 0:
        s = evaluate(state, player_id, 'defensive')
        node = TreeNode("Leaf", "LEAF", s) if tree_build else None

        return s, None, node

    
    if current_player in skipped:#skip propagation
        new_skipped = skipped - {current_player}
        index = PLAYERS.index(current_player)
        next_index = PLAYERS[(index + 1) % len(PLAYERS)]
        
        return minimax(state, depth, player_id, next_index, new_skipped,alpha, beta, tree_build)

    
    valid   = get_valid_moves(state[current_player], state['top_card'])
    if valid:
        actions = valid
    else:
        actions = [None] 

    is_max    = (current_player == player_id)
    if is_max:
        node_type = "MAX"
    else:
        node_type = "MIN"
    
    index = PLAYERS.index(current_player)
    next_index = PLAYERS[(index + 1) % len(PLAYERS)]

    root_node = (TreeNode(f"{current_player.upper()} | top={state['top_card']}",  node_type) if tree_build else None)

    if is_max:
        bestscore = float('-inf')
    else:
        bestscore = float('inf')
        
    if actions:
        bestmove = actions[0]
    else:
        bestmove = None

    for i in actions:
        state_updated, skip_p = apply_move(state, (current_player, i))
        if skip_p:
            new_skipped = {skip_p}
        else:
            new_skipped = set()

        child_score, _, child_node = minimax(state_updated, depth - 1, player_id, next_index, new_skipped,alpha, beta, tree_build)

        if tree_build:
            if i is not None:
                action_str = str(i)
            else:
                action_str = "Draw forcefully"
            anode = TreeNode(action_str, "ACTION", child_score)
            if child_node:
                anode.add_child(child_node) 
            root_node.add_child(anode)

        if is_max:
            if child_score > bestscore:
                bestscore = child_score
                bestmove  = i
            alpha = max(alpha, bestscore)
            
        else:
            if child_score < bestscore:
                bestscore = child_score
                bestmove  = i
                
            beta = min(beta, bestscore)

        
        # alphabeta pruning
        if beta <= alpha:
            if tree_build:
                root_node.add_child(TreeNode("(pruned)", "PRUNED"))
            break



    if tree_build:
        root_node.score = bestscore
    return bestscore, bestmove, root_node

In [167]:
def expectimax(state,depth,player_id,current_player,skipped, tree_build= True):

    for p in PLAYERS:
        if len(state[p]) == 0:
            if p == player_id:
                s = 1000.0 + depth
            else:
                s = -1000.0 - depth
            node = TreeNode(f"WIN({p.upper()})", "TERMINAL", s) if tree_build else None
            return s, None, node

    if depth == 0:
        s = evaluate(state, player_id, 'offensive')
        node = TreeNode("Leaf", "LEAF", s) if tree_build else None
        return s, None, node

    
    if current_player in skipped:
        new_skipped = skipped - {current_player}
        index = PLAYERS.index(current_player)
        nextindex = PLAYERS[(index + 1) % len(PLAYERS)]
        return expectimax(state, depth, player_id, nextindex, new_skipped, tree_build)

    valid = get_valid_moves(state[current_player], state['top_card'])
    index   = PLAYERS.index(current_player)
    nextindex   = PLAYERS[(index + 1) % len(PLAYERS)]

    
    if current_player == player_id:
        root_node = (TreeNode(f"{current_player.upper()} | top={state['top_card']}", "MAX")if tree_build else None)
        bestscore = float('-inf')
        bestmove  = None

        for card in valid:
            state_updated,skip_p = apply_move(state, (current_player, card))
            if skip_p:
                new_skipped = {skip_p}
            else:
                new_skipped = set()
                        
            score, _, child_node = expectimax(state_updated, depth - 1, player_id, nextindex, new_skipped, tree_build)

            if tree_build:
                anode = TreeNode(f"Play {card}", "ACTION", score)
                if child_node:
                    anode.add_child(child_node)  
                root_node.add_child(anode)

            if score > bestscore:
                bestscore = score
                bestmove  = card

        if not valid:
            if state['deck']:
                chance_score, chance_node = _chance_node(
                    state, current_player, player_id,
                    nextindex, set(), depth, tree_build)
                if tree_build:
                    root_node.add_child(chance_node)
                if chance_score > bestscore:
                    bestscore = chance_score
                    bestmove  = None
                    
            else:
                bestscore = evaluate(state, player_id, 'offensive')

        if tree_build:
            root_node.score = bestscore

            
        return bestscore, bestmove, root_node


    
    else:
        root_node = (TreeNode( f"{current_player.upper()} | top={state['top_card']} ","OPP") if tree_build else None)

        if valid:
            actions = valid
        else:
            if state['deck']:
                actions = [None]   
            else:
                actions = []       

        if not actions:
            s = evaluate(state, player_id, 'offensive')
            if tree_build:
                root_node.score = s
                
            return s, None, root_node

        total_score = 0.0
        
        for move in actions:
            state_updated,skip_p = apply_move(state, (current_player, move))
            if skip_p:
                new_skipped = {skip_p}
            else:
                new_skipped = set()
            
            score, _, child_node = expectimax(state_updated, depth - 1, player_id, nextindex, new_skipped, tree_build)
            
            total_score += score

            if tree_build:
                if move is not None:
                    action_str = str(move)
                else:
                    action_str = "Draw (forced)"
                anode = TreeNode(action_str, "ACTION", score)
                if child_node:
                    anode.add_child(child_node)   
                root_node.add_child(anode)

        expected_score = total_score / len(actions)

        
        if tree_build:
            root_node.score = expected_score

            
        return expected_score, None, root_node



In [168]:
def _chance_node(state, current_player,  player_id,next_player, skipped, depth, tree_build):
    
    deck = state['deck']
    
    if tree_build:
        chance_node = TreeNode("Draw Chance", "CHANCE")
    else:
        chance_node = None

    if not deck:
        s = evaluate(state, player_id, 'offensive')
        
        if tree_build:
            chance_node.score = s
        return s, chance_node

    card_counts = {}
    for c in deck:
        key = (c.colour, c.value)
        
        if key in card_counts:
            card_counts[key] += 1
        else:
            card_counts[key] = 1
    

    deck_size = len(deck)
    expected_val = 0.0

    for (colour, value), count in card_counts.items():
        prob  = count / deck_size
        drawn = Card(colour, value)

        state_updated = copy.deepcopy(state)
        state_updated[current_player].append(drawn)
        
        for i, c in enumerate(state_updated['deck']):
            if c == drawn:
                state_updated['deck'].pop(i)
                break

        score, _, child_node = expectimax(state_updated, depth - 1, player_id, next_player, skipped, tree_build)
        expected_val += prob * score

        if tree_build:
            cnode = TreeNode(f"Draw {drawn} (p={prob:.2f})", "  CHANCE", score)
            if child_node:
                cnode.add_child(child_node)   
            chance_node.add_child(cnode)

    if tree_build:
        chance_node.score = expected_val

        
    return expected_val, chance_node

In [169]:
class Game_UNO:
    
    def __init__(self, mode = 'simulation', seed = 42):
        random.seed(seed)
        self.mode = mode
        self.seed = seed         

        deck = deck_generator()
        self.state = {'p1':       [deck.pop() for _ in range(5)],
            'p2':       [deck.pop() for _ in range(5)],
            'p3':       [deck.pop() for _ in range(5)],
            'top_card': deck.pop(),
            'deck':     deck,
            'discards': [],
            
        }

        
        attempts = 0
        while (self.state['top_card'].is_skip() and self.state['deck'] and attempts < 10):
            self.state['deck'].insert(0, self.state['top_card'])
            self.state['top_card'] = self.state['deck'].pop()
            attempts += 1

        self.current_player = 'p1'
        self.skipped = set()
        self.winner = None
        self.move_history = []
        self.turn = 0
        self.game_over = False
        

        self.display()


    def display(self):
        print("\n" + "-" * 65)
        print("                       UNO                    ")
        print("-" * 65)
        print(f"  Mode : {'Simulation (AI all players)' if self.mode == 'simulation' else 'Manual (You are P3)'}")
        
        print(f"  Seed : {self.seed}")   
        print(f"\n  Top Card  : {self.state['top_card']}")
        
        for p in PLAYERS:
            print(f"  {p.upper()} hand: {self.state[p]}")
        print(f"  Deck size : {len(self.state['deck'])} cards")
        
        print("_" * 65)

    def show_state(self):
        disc = len(self.state.get('discards', []))
        print(f"\n{'-'*65}")
        print(f"  Turn {self.turn:3d} | {self.current_player.upper()}'s move" f"  Top: {self.state['top_card']}")
        print(f"{'-'*65}")
        for p in PLAYERS:
            if p == self.current_player:
                marker = " <"
            else:
                marker = "  "
            
            print(f"  {PLAYER_NAMES[p]:<32} {len(self.state[p]):2d} cards  {self.state[p]}{marker}")

            
        print(f"  Deck: {len(self.state['deck'])} | Discards: {disc}", end="")
              
        if self.skipped:
            print(f"  | Skipped: {self.skipped}", end="")

            
        print()

   
    def show_depth1_scores(self, player: str, strategy: str):
       
        valid   = get_valid_moves(self.state[player], self.state['top_card'])
        if valid:
            actions = valid
        else:
            actions = [None]

        if strategy == 'defensive':
            label = "Heuristic score"
        else:
            label = "Expected score"

        print(f"\n  Top card: {self.state['top_card']}")
        print(f"  {player.upper()} hand:")
        
        for c in self.state[player]:
            print(f"  {c}")
            
            
        print(f"\n  All actions considered at depth 1:")
        
        for action in actions:
            tmp, _ = apply_move(self.state,( player, action))
            score  = evaluate(tmp, player, strategy)
            if action is not None:
                label1 = str(action)
            else:
                label1 = "Draw (forcefully )"
                
            print(f"    {label1:<26}  {label}: {score:.1f}")

    
    def _p1_move(self):
        print("\n P1 MINIMAX DEFENSIVE")
        self.show_depth1_scores('p1', 'defensive')

        score, bestmove, tree = minimax(self.state, depth=3,player_id='p1', current_player='p1',skipped=self.skipped, alpha=float('-inf'), beta=float('inf'),tree_build=True)

        if bestmove is not None:
            label = str(bestmove)
        else:
            label = "Draw (forcefullyu)"
        
        print(f"\n  -- Minimax decision   depth 3: {label}  / score = {score:.1f}")
        print("\n  Minimax search tree (depth 2):   ")
        
        if tree:
            print_tree_smart(tree, max_depth=3)
        print()

        
        return bestmove

    
    def _p2_move(self) :
        print("\n  P2 EXPECTIMAX OFFENSIVE")
        valid = get_valid_moves(self.state['p2'], self.state['top_card'])

        print(f"\n  Top card: {self.state['top_card']}")
        print("  P2 hand:")
        
        for c in self.state['p2']:
            print(f"  {c}")
        print("\n  All actions considered at depth 1:")

        if valid:
            for card in valid:
                tmp, _ = apply_move(self.state, ('p2', card))
                s = evaluate(tmp, 'p2', 'offensive')
                
                print(f" Play {str(card):<22}  heuristic score: {s:.1f}")
        else:
            if self.state['deck']:
                cs, _ = _chance_node(self.state, 'p2', 'p2', 'p3', set(), 1, False)
                print(f" Draw (forced):   heuristic score: {cs:.2f}")

        score, bestmove, tree = expectimax(self.state, depth=3,player_id='p2', current_player='p2', skipped=self.skipped,tree_build=True)

        if bestmove is not None:
            label = str(bestmove)
        else:
            label = "Draw (forcefully)"
            
        print(f"\n -- Expectimax decision (depth 3): {label}  /  score = {score:.1f}")
        print("\n Expectimax search tree (depth-2 ):")
        if tree:
            print_tree_smart(tree, max_depth=3)
            
        print()
        
        return bestmove

        

   
    def _p3_manual(self):
        print("\n  P3 HUMAN or YOU")
        print(f"Your hand : {self.state['p3']}")
        print(f"Top card  : {self.state['top_card']}")
        valid = get_valid_moves(self.state['p3'], self.state['top_card'])

        if not valid:
            print(" NO valid moves u must draw a card")
            return None

        print(" Valid moves:")
        for i, card in enumerate(valid):
            print(f"    {i}: {card}")
        

        while True:
            try:
                choice = int(input("  Enter index: "))
                if 0 <= choice < len(valid):
                    return valid[choice]
                print("Invalid Try again")
                
            except ValueError:
                print("Enter a number")

    
    def _p3_sim(self):
        print("\n P3 MINIMAX SIMULATION")
        self.show_depth1_scores('p3', 'defensive')

        score, bestmove, _ = minimax(self.state, depth=3, player_id='p3', current_player='p3', skipped=self.skipped, alpha=float('-inf'), beta=float('inf'), tree_build=False)

        if bestmove is not None:
            label = str(bestmove)
        else:
            label = "Draw (forcefully)"
        print(f"\n  --- Minimax decision (depth 3): {label}  |  score = {score:.1f}")

        
        return bestmove

    
    def _run(self, player, card):
        old_len = len(self.state[player])
        self.state, skip_p = apply_move(self.state, (player, card))

        if card is None:
            if len(self.state[player]) > old_len:
                drawn = self.state[player][-1]
                print(f"\n  >> {player.upper()} DRAWS -> {drawn}")
            else:
                print(f"\n  >> {player.upper()} tried to draw but deck is empty!")
                
        else:
            print(f"\n  >> {player.upper()} PLAYS -> {card}")

        if skip_p:
            self.skipped.add(skip_p)
            print(f"  [SKIP] {skip_p.upper()} will be SKIPPED next turn!!! womp")

        self.move_history.append({'turn':     self.turn,
            'player':   player,
            'action':   str(card) if card is not None else 'Draw',
            'top_card': str(self.state['top_card']),})

    
    def play_turn(self):
        self.turn += 1
        player = self.current_player
        self.show_state()

        
        if player in self.skipped:#skip
            print(f"\n  [SKIP] {player.upper()} is SKIPPED this turn!!! womp")
            self.skipped.discard(player)
            index = PLAYERS.index(player)
            
            self.current_player = PLAYERS[(index + 1) % len(PLAYERS)]
            return

    
        if player == 'p1':
            card = self._p1_move()
        elif player == 'p2':
            card = self._p2_move()
        else:
            if self.mode == 'manual':
                card = self._p3_manual()
            else:
                card = self._p3_sim()

        self._run(player, card)

        
        for p in PLAYERS:
            if len(self.state[p]) == 0:
                self.game_over = True
                self.winner    = p
                print(f"\n{'_'*65}")
                print(f"  GAME OVER {PLAYER_NAMES[p]} WINSSS!")
                print("-" * 65)
                return

        index = PLAYERS.index(player)
        self.current_player = PLAYERS[(index + 1) % len(PLAYERS)]

   
    def play(self, max_turns = 200):#complete game
        while not self.game_over and self.turn < max_turns:
            self.play_turn()
            
            if self.mode == 'manual' and not self.game_over:
                input("\nPRESS ENTER TO NEXT TURN ")

        if not self.game_over:
            print(f"\n  Game ended after {max_turns} turns (no winner) WOMPP")
            
            for p in PLAYERS:
                print(f" {p.upper()}: {len(self.state[p])} cards")


        
        return self.winner

In [170]:
import io, contextlib

In [171]:
def compare_algorithms(n_games=5, randomize=False):
    import io, contextlib

    print("\n" + "_" * 65)
    if randomize:
        mode_label = "RANDOMIZED"
    else:
        mode_label = "FIXEDSEED"
    print(f"  ALGORITHM COMPARISON  ({n_games} games)  [{mode_label}]")
    print("_" * 65)

    wins = {'p1': 0, 'p2': 0, 'p3': 0, 'timeout': 0}

    for i in range(n_games):
        if randomize:
            seed = random.randint(1, 1000000)
        else:
            seed = i * 137 + 4

        buf = io.StringIO()
        
        with contextlib.redirect_stdout(buf):
            game = Game_UNO(mode='simulation', seed=seed)
            winner = game.play(max_turns=150)

        if winner:
            wins[winner] += 1
            print(f"Game {i+1}/{n_games}  (seed={seed})  Winner: {PLAYER_NAMES[winner]}")
        else:
            wins['timeout'] += 1
            print(f"Game {i+1}/{n_games}  (seed={seed})  Winner: No winner (timeout)")


    
    print("\n" + "-" * 65)
    print("  RESULTS")
    print("-" * 65)
    print(f"P1 Minimax (Defensive)     : {wins['p1']:2d} / {n_games}")
    print(f"P2 Expectimax (Offensive)  : {wins['p2']:2d} / {n_games}")
    print(f"P3 Minimax (Simulation)    : {wins['p3']:2d} / {n_games}")
    print(f"Timeout                    : {wins['timeout']:2d} / {n_games}")

    best = max(['p1', 'p2', 'p3'], key=lambda p: wins[p])

    if wins[best] == 0:
        verdict = "No clear winner, all games timed out"
    elif best == 'p1':
        verdict = "P1 Minimax Defensive performed best"
    elif best == 'p2':
        verdict = "P2 Expectimax Offensive performed best"
    else:
        verdict = "P3 Minimax Simulation performed best"

    print(f"\nConclusion: {verdict}")

    
    return wins

In [172]:
def tree_demo():
    
    print("\n" + "_" * 65)
    print("SEARCH TREE DEMONSTRATION")
    print("_" * 65)

    ex = {'p1': [Card('Red', 3), Card('Blue', 7), Card('Red', 'Skip')],
        'p2': [Card('Green', 5), Card('Yellow', 7), Card('Blue', 2)],
        'p3': [Card('Blue', 9), Card('Red', 9), Card('Green', 1)],
        'top_card': Card('Red', 7),
        'deck':  [Card('Blue', 4), Card('Green', 6),Card('Yellow', 3), Card('Red', 1), Card('Green', 3)],
        'discards': [],}

    print(f"Deck ({len(ex['deck'])}): {ex['deck']}")
    print(f"Top card :{ex['top_card']}", f"\n P1 hand: {ex['p1']}")
    print(f"P2 hand: {ex['p2']}")
    print(f"P3 hand: {ex['p3']}")
    

   
    print("\n" + "-" * 65)
    print("p1  MINIMAX DEFENSIVE  depth=3")
    print("-" * 65)

    valid_p1  = get_valid_moves(ex['p1'], ex['top_card'])
    if valid_p1:
        actions_p1 = valid_p1
    else:
        actions_p1 = [None]

    print(f"\n Top card: {ex['top_card']}")
    
    print("P1 hand:")
    for c in ex['p1']:
        print(f"    {c}")
        
    print("\n All actions at depth 1  (Heuristic score):") 
    
    for action in actions_p1:
        tmp, _ = apply_move(ex, ('p1', action))
        s = evaluate(tmp, 'p1', 'defensive')
        
        if action is not None:
            label = str(action)
        else:
            label = "Draw (forcefully)"
            
        print(f"    {label:<24}  Heuristic score: {s:.1f}")

    score_mm, move_mm, tree_mm = minimax(ex, depth=3, player_id='p1', current_player='p1', skipped=set(), tree_build=True)

    print(f"\nBest Move : {move_mm if move_mm is not None else 'Draw forced' }")
    print(f"Score     : {score_mm:.1f}")
    print("\n  Minimax Search Tree (full depth 3):")
    
    if tree_mm:
        print_tree_smart(tree_mm, max_depth=3)

   
    print("\n" + "-" * 65)
    print("p2 EXPECTIMAX OFFENSIVE  depth=3")
    print("-" * 65)

    valid_p2 = get_valid_moves(ex['p2'], ex['top_card'])

    print(f"\nTop card: {ex['top_card']}")
    print("P2 hand:")
    
    for c in ex['p2']:
        print(f"    {c}")
        
    print("\n All actions at depth 1  (heuristic score):")
    
    if valid_p2:
        for card in valid_p2:
            tmp, _ = apply_move(ex, ( 'p2', card))
            s = evaluate(tmp, 'p2', 'offensive')
            
            print(f"Play {str(card):<20}  heuristic score: {s:.1f}")
    else:
        if ex['deck']:
            cs, _ = _chance_node(ex, 'p2', 'p2', 'p3', set(), 1, False)
            print(f"    Draw forced:       heuristic score: {cs:.2f}")

    score_ex, move_ex, tree_ex = expectimax(ex, depth=3, player_id='p2', current_player='p2',skipped=set(), tree_build=True)

    print(f"\n  Best Move : {move_ex if move_ex is not None else 'Draw forced'}")
    print(f"  Score     : {score_ex:.1f}")
    print("\nExpectimax search tree full depth 3:  ")
    
    if tree_ex:
        print_tree_smart(tree_ex, max_depth=3)


In [176]:
print("\n" + "_" * 65)

print("\n                 UNO — MENU            ")
print("_" * 65)
print("  1. Simulation Mode- All 3 players are AI")
print("  2. Manual Mode  - You play as Player 3")
print("  3. Compare  - 5 games + algorithm analysis")

print("_" * 65)

try:
    choice = input("\n  Enter choice [1-3] (default=1): ").strip() or '1'
except (EOFError, KeyboardInterrupt):
    choice = '1'

if choice == '1':
    game = Game_UNO(mode='simulation', seed=random.randint(1, 1000000))
    game.play(max_turns=150)

elif choice == '2':
    game = Game_UNO(mode='manual', seed=random.randint(1, 1000000))
    game.play(max_turns=150)
    
elif choice == '3':
    compare_algorithms(n_games=50 , randomize = False)


else:
    print("Invalid choice Running simulation as defualt")
    game = Game_UNO(mode='simulation', seed=random.randint(1, 1000000))
    game.play(max_turns=150)





_________________________________________________________________

                 UNO — MENU            
_________________________________________________________________
  1. Simulation Mode- All 3 players are AI
  2. Manual Mode  - You play as Player 3
  3. Tree Demo - Search tree on fixed example
  4. Compare  - 5 games + algorithm analysis
_________________________________________________________________



  Enter choice [1-5] (default=1):  2



-----------------------------------------------------------------
                       UNO                    
-----------------------------------------------------------------
  Mode : Manual (You are P3)
  Seed : 664061

  Top Card  : Blue 6
  P1 hand: [Green 2, Green 4, Green Skip, Green 7, Yellow 0]
  P2 hand: [Green 3, Yellow 1, Blue 1, Red Skip, Green 0]
  P3 hand: [Red 6, Yellow 5, Green 6, Yellow 4, Red 4]
  Deck size : 28 cards
_________________________________________________________________

-----------------------------------------------------------------
  Turn   1 | P1's move  Top: Blue 6
-----------------------------------------------------------------
  P1 (Minimax Defensive)            5 cards  [Green 2, Green 4, Green Skip, Green 7, Yellow 0] <
  P2 (Expectimax Offensive)         5 cards  [Green 3, Yellow 1, Blue 1, Red Skip, Green 0]  
  P3                                5 cards  [Red 6, Yellow 5, Green 6, Yellow 4, Red 4]  
  Deck: 28 | Discards: 0

 P1 MINIMAX D


PRESS ENTER TO NEXT TURN  



-----------------------------------------------------------------
  Turn   2 | P2's move  Top: Blue 6
-----------------------------------------------------------------
  P1 (Minimax Defensive)            6 cards  [Green 2, Green 4, Green Skip, Green 7, Yellow 0, Blue 2]  
  P2 (Expectimax Offensive)         5 cards  [Green 3, Yellow 1, Blue 1, Red Skip, Green 0] <
  P3                                5 cards  [Red 6, Yellow 5, Green 6, Yellow 4, Red 4]  
  Deck: 27 | Discards: 0

  P2 EXPECTIMAX OFFENSIVE

  Top card: Blue 6
  P2 hand:
  Green 3
  Yellow 1
  Blue 1
  Red Skip
  Green 0

  All actions considered at depth 1:
 Play Blue 1                  heuristic score: 48.5

 -- Expectimax decision (depth 3): Blue 1  /  score = 48.5

 Expectimax search tree (depth-2 ):
       P2-AI (+48.5)
             |
    Play Blue 1 (+48.5)
             |
      P3-OPP (+48.5)
             |
   Draw (forced) (+48.5)
         (1 more)


  >> P2 PLAYS -> Blue 1


...


-----------------------------------------------------------------
  Turn  30 | P3's move  Top: Blue 5
-----------------------------------------------------------------
  P1 (Minimax Defensive)            1 cards  [Yellow 0]  
  P2 (Expectimax Offensive)         2 cards  [Yellow 1, Yellow Skip]  
  P3                                5 cards  [Red 6, Yellow 5, Yellow 4, Red 4, Red 3] <
  Deck: 18 | Discards: 17

  P3 HUMAN or YOU
Your hand : [Red 6, Yellow 5, Yellow 4, Red 4, Red 3]
Top card  : Blue 5
 Valid moves:
    0: Yellow 5


  Enter index:  0



  >> P3 PLAYS -> Yellow 5



PRESS ENTER TO NEXT TURN  



-----------------------------------------------------------------
  Turn  31 | P1's move  Top: Yellow 5
-----------------------------------------------------------------
  P1 (Minimax Defensive)            1 cards  [Yellow 0] <
  P2 (Expectimax Offensive)         2 cards  [Yellow 1, Yellow Skip]  
  P3                                4 cards  [Red 6, Yellow 4, Red 4, Red 3]  
  Deck: 18 | Discards: 18

 P1 MINIMAX DEFENSIVE

  Top card: Yellow 5
  P1 hand:
  Yellow 0

  All actions considered at depth 1:
    Yellow 0                    Heuristic score: 56.0

  -- Minimax decision   depth 3: Yellow 0  / score = 1002.0

  Minimax search tree (depth 2):   
     P1-AI (+1002.0)
            |
   Yellow 0 (+1002.0)
           /
        WIN(P1)



  >> P1 PLAYS -> Yellow 0

_________________________________________________________________
  GAME OVER P1 (Minimax Defensive) WINSSS!
-----------------------------------------------------------------


<h1>Generated search tree (fixed example)</h1>

In [178]:
tree_demo()


_________________________________________________________________
SEARCH TREE DEMONSTRATION
_________________________________________________________________
Deck (5): [Blue 4, Green 6, Yellow 3, Red 1, Green 3]
Top card :Red 7 
 P1 hand: [Red 3, Blue 7, Red Skip]
P2 hand: [Green 5, Yellow 7, Blue 2]
P3 hand: [Blue 9, Red 9, Green 1]

-----------------------------------------------------------------
p1  MINIMAX DEFENSIVE  depth=3
-----------------------------------------------------------------

 Top card: Red 7
P1 hand:
    Red 3
    Blue 7
    Red Skip

 All actions at depth 1  (Heuristic score):
    Red 3                     Heuristic score: 48.0
    Blue 7                    Heuristic score: 48.0
    Red Skip                  Heuristic score: 44.0

Best Move : Red Skip
Score     : 49.0

  Minimax Search Tree (full depth 3):
                                 P1-AI (+49.0)
              /___________________________\________________________\
        Red 3 (+48.0)              Blue 7 (

<h1>THEORATICAL:</h1>

<h1>EVALUATION FUNCTION EXPLANATION</h1>

Base formula (from assignment) Score = 50 − 5x(CAI) + 2x(Copp) + 3x(S)
 - CAI  = number of cards in the AI player's hand  (lower -> better)
 - Copp = average cards held by the two opponents   (higher -> better)
 - S    = number of Skip cards in the AI's hand     (higher -> better)

The formula tuned differently per strategy

DEFENSIVE (Player 1 minimax)
  - owncard penalty raised  (−6 instead of −5) defensive play is obsessed with keeping the hand small
  - skip reward raised (+4 instead of +3) skip cards are gold they freeze opponents at critical moments
  - opponent burden kept moderate (+2) we care less about loading opponents and more about protecting ourself
  Formula -> 50 − 6xCAI + 2xCopp + 4xS



OFFENSIVE (P2 expectimax)
  - owncard penalty standard (−5)we still want to shed cards but not at the expense of loading opponents
  - opponent burden raised (+3 instead of +2)offensive play tries to force opponents to have MORE cards (draw situations)
  -skip reward lowered (+2)skips are useful but we prefer getting rid of cards over delaying opps by skipping them
  Formula -> 50 − 5xCAI + 3xCopp + 2xS

#### <h1>ALGORITHM ANALYSIS</h1>

<h3>BOTH PROPERTIES</h3>

**minimax defensive  (P1 & P3)**
Strategy  : prevent opponents from winning save skips for crises
Assumption: all opponents play optimally against us (worst case)
Search    : alpha-beta pruning prunes branches when beta <= alpha
Weights   : -6*own  +2*opp  +4*skip
Strength  : robust when opponents are smart
Weakness  : overly pessimistic when opponents play randomly may stall because it holds skips too long

**expectimax offensive  (P2)**
Strategy  : aggressive card-shedding; exploit the draw deck
Assumption: draw is a CHANCE NODE with exact deck probabilities ppponents modelled as average over ALL legal moves (deterministic expected value not random sampling)
Weights   : -5*own  +3*opp  +2*skip
Strength  : explicitly models randomness best fit for UNO
weakness  : opponent "average" assumption may be optimistic


<h2>DETAILED ANALYSIS</h2>

player 1 uses minimax with a defensive strategy. player 2 uses expectimax with an offensive strategy. player 3 has two possible modes. In manual mode player 3 is controlled by the user in simulation mode player 3 also uses minimax. The purpose of this comparison was to see how both search methods behave in the same game and to understand which one works better in this simplified UNO setting

the strategy of player 1 is defensive this means player 1 is not only trying to finish the hand quickly but is also trying to stay safe against bad future moves. minimax works by assuming that the other players will always do the move that is worst for the current AI because of this player 1 becomes careful. It values having fewer cards in hand and it also values Skip cards more because Skip can stop the next player and can prevent an opponent from getting close to winning this makes minimax a cautious player it tries to avoid risky moves and it tries to protect itself from the worst possible future

the strategy of player 2 is offensive this player uses expectimax instead of assuming that everything will go in the worst possible way it tries to calculate what is likely to happen on average. This is useful because UNO has randomness in it due to the draw deck if a player has no valid move then that player must draw and the card that comes out is uncertain. expectimax handles this by treating the draw as a chance event. In simple words it looks at the possible cards that can come from the deck and thinks about their chances. This makes player 2 more aggressive it focuses more on dropping cards quickly and trying to reach zero cards first still cares about skip cards but not as much as minimax does because its main goal is fast progress

When I tested both algorithms I noticed there was not only one fixed winner in every situation. The results changed depending on whether the games were run with fixed seeds or random seeds this makes sense because UNO is a game with chance the order of the deck matters a lot small changes in the starting hands or in the future draws can completely change who gets the advantage
So if someone asks 
**which algorithm is best the honest answer is that it depends on the exact setting and on how much randomness is affecting the game**.

- In the fixed seed comparison the result was very close between all players player 2 using expectimax won slightly more games than the othersplayer 3 using minimax was very close behind and player 1 using minimax also won many games this shows that in those specific deck orders expectimax got a small advantage **The reason is that expectimax is made to deal with chance and it can benefit when the deck arrangement allows an aggressive plan to work well since UNO has a draw mechanism this can help expectimax because it is built to reason about uncertain future cards**

- In the randomized comparison the result changed. player 3 using minimax won the most games player 1 also did better than player 2 in that set this shows that across a wider variety of random deck orders minimax can be very strong especially for player 3  1 major reason is turn order. **player 3 plays after player 1 and player 2. That means player 3 gets to react after seeing more of what happened in the round gives more information and often makes defensive play stronger. Since player 3 also uses minimax in simulation mode it combines careful search with the benefit of moving later that is why player 3 can perform very well**

- Another reason for this behavior is that the opponents in my game are not random weak players player 1 and player 3 both use minimax they are making smart choices expectimax usually works best when uncertainty in the environment is the main challenge in this game however player 2 is surrounded by opponents that are also planning well so even though expectimax is good at handling chance it can still lose if the other players make strong defensive moves and use their turns well this is especially true when the game becomes more about timing and blocking rather than only about probability

**In general for UNO we would expect expectimax to perform well because UNO includes randomness through drawing cards**
A method that can model chance should logically have an advantage when the deck matters at the same time we would also expect minimax to perform well when the opponents are smart and when defensive timing matters in this simplified version of UNO there are only number cards and skip cards there are no wild cards and no draw two or draw four cards because of this the game is simpler and more controlled than full UNO. that means defensive search can stay very effective **So in this version it is normal that minimax remains competitive and in some cases even stronger than expectimax**

Another thing I noticed is that immediate good moves are not always the best long term moves sometimes a move can look weaker at depth one but later become the best move after looking deeper into the tree this happened in minimax because it searches future turns and sees how opponents may respond. This is one of the main strengths of adversarial search. It does not only judge the move based on the current momentit tries to understand what will happen after that move this is why deeper search can choose a move that looks surprising at first

The difference between player 1 and player 3 is also important both use minimax but they do not always get the same results this means the algorithm itself is not the only factor turn position also matters player 1 goes first so it has to act with less information player 3 goes later so it can often react better This is why player 3 can outperform player 1 even though they use the same search method soo **when comparing results should not only say minimax is better or expectimax is better that player order has an effect on the final outcome**

My final verdict is that both algorithms worked correctly and both showed useful strengths expectimax was slightly better in the fixed seed experiment which suggests that the offensive probability based strategy can be very effective when the deck order suits it. minimax simulation through player 3 was better in the randomized experiment which suggests that defensive search plus later turn order can be even stronger over many changing situations so the best overall conclusion is not that one algorithm always dominates he better conclusion is that **expectimax is strong when randomness plays in its favor while minimax is strong when careful defensive planning and position in the turn order matter more**

If I had to give one plain final answer I would say that in this simplified UNO game expectimax is the better offensive idea and minimax is the better defensive idea expectimax is better at handling uncertainty from the deck and minimax is better at protecting itself against strong opponents in actual results from my program player 2 did slightly better in the fixed comparison but player 3 did best in the randomized comparison. Because of that the most honest conclusion is that performance depends on the game setup and the deck order stilll both algorithms clearly show different thinking styles and both are valid for UNO